# LeetCode 706 — Design HashMap bằng Python

**Link đề:** https://leetcode.com/problems/design-hashmap/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Linked List, Design, Hash Function

```python
class MyHashMap:

    def __init__(self):
        ...

    def put(self, key: int, value: int) -> None:
        ...

    def get(self, key: int) -> int:
        ...

    def remove(self, key: int) -> None:
        ...
```

## Mục tiêu học
1. Hiểu `dict` bên trong: **key → bucket (băm) → tìm cặp `(key, value)` trong bucket**.
2. Phân biệt rõ hai trường hợp của `put`: **thêm mới** và **cập nhật** key đã có.
3. Cài 4 cách không dùng `dict`: list các cặp → mảng trực tiếp → **băm + chaining các cặp `[key, value]`** → băm + linked list.
4. Trace trạng thái bucket sau từng lệnh, chứng minh invariant "mỗi key xuất hiện tối đa một lần".
5. Phân tích Time/Space trung bình và xấu nhất, nhận dạng khi nào dùng mảng đếm thay cho dict.

## 1. Đề bài và ví dụ

Thiết kế HashMap **không dùng thư viện bảng băm có sẵn**:
- `MyHashMap()`: khởi tạo map rỗng.
- `put(key, value)`: thêm cặp `(key, value)`. Nếu `key` **đã có**, **cập nhật** value.
- `get(key)`: trả value của `key`, hoặc `-1` nếu không có.
- `remove(key)`: xóa `key` (và value) nếu có.

| Lệnh | Map sau lệnh | Trả về |
|---|---|:---:|
| `put(1, 1)` | `{1:1}` | null |
| `put(2, 2)` | `{1:1, 2:2}` | null |
| `get(1)` | | `1` |
| `get(3)` | | `-1` |
| `put(2, 1)` | `{1:1, 2:1}` (cập nhật) | null |
| `get(2)` | | `1` |
| `remove(2)` | `{1:1}` | null |
| `get(2)` | | `-1` |

**Constraints:** `0 <= key, value <= 10^6`; tối đa `10^4` lời gọi `put`, `get`, `remove`.

Chú ý: value có thể là `0`, nên **không** được dùng `0` để biểu diễn "không có". Đề chọn `-1` vì value không âm.

In [1]:
OPS = ["MyHashMap", "put", "put", "get", "get", "put", "get", "remove", "get"]
ARGS = [[], [1, 1], [2, 2], [1], [3], [2, 1], [2], [2], [2]]
EXP = [None, None, None, 1, -1, None, 1, None, -1]

def run_ops(cls, ops, args):
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == ops[0]:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print(len(OPS), "lệnh; kết quả mong đợi:", EXP)

9 lệnh; kết quả mong đợi: [None, None, None, 1, -1, None, 1, None, -1]


`run_ops` chạy kịch bản giống LeetCode: lệnh đầu tạo đối tượng, các lệnh sau gọi phương thức theo tên bằng `getattr`. `cls(*a)` "mở" list tham số thành đối số.

## 2. Nền tảng: `dict` của Python làm gì?

Dùng `dict` có sẵn **chỉ để đối chiếu** hành vi (không dùng trong lời giải):

In [2]:
d = {}
d[1] = 1
d[2] = 2
d[2] = 1                 # gán lại = cập nhật
print(d, d.get(3, -1))   # get có giá trị mặc định
d.pop(2, None)           # xóa, không lỗi nếu không có
print(d)

{1: 1, 2: 1} -1
{1: 1}


### 2.1. Tuple / list hai phần tử để lưu cặp `(key, value)`

Ta sẽ lưu mỗi cặp là **list** `[key, value]` (list sửa được, tuple thì không). Unpack: `k, v = pair`.

In [3]:
pair = [2, 2]
k, v = pair
print("key =", k, "| value =", v)
pair[1] = 1              # cập nhật value tại chỗ
print(pair)
t = (2, 2)
try:
    t[1] = 1
except TypeError as e:
    print("tuple không sửa được:", e)

key = 2 | value = 2
[2, 1]
tuple không sửa được: 'tuple' object does not support item assignment


### 2.2. Hàm băm `key % M` và bucket

Gấp miền key lớn (`0..10^6`) vào `M` bucket. Các key cùng số dư **va chạm** và dùng chung bucket.

In [4]:
M = 5
for key in [1, 6, 11, 2, 1000000]:
    print(f"key {key:>7} -> bucket {key % M}")

key       1 -> bucket 1
key       6 -> bucket 1
key      11 -> bucket 1
key       2 -> bucket 2
key 1000000 -> bucket 0


### 2.3. Tạo list các bucket độc lập

`[[] for _ in range(M)]` tạo `M` list **riêng biệt**. Không dùng `[[]] * M` (tạo `M` tham chiếu tới cùng một list).

In [5]:
buckets = [[] for _ in range(3)]
buckets[0].append([7, 70])
print(buckets)
print("id khác nhau:", len({id(x) for x in buckets}) == 3)

[[[7, 70]], [], []]
id khác nhau: True


### 2.4. Duyệt có chỉ số để xóa: `enumerate` + `pop(i)`

Để xóa cặp có key cho trước trong bucket, tìm **vị trí** `i` rồi `bucket.pop(i)`.

In [6]:
bucket = [[1, 10], [6, 60], [11, 110]]
for i, (k, v) in enumerate(bucket):
    if k == 6:
        bucket.pop(i)
        break
print(bucket)

[[1, 10], [11, 110]]


`for i, (k, v) in enumerate(bucket)`: mỗi phần tử là `[k, v]`, ta unpack lồng ngay trong vòng `for`. `break` thoát vòng ngay sau khi xóa — **quan trọng**, vì sửa list trong lúc đang duyệt nó là nguy hiểm.

## 3. Cách 1 — List các cặp (baseline)

Giữ một list `[[key, value], ...]`, mọi thao tác **duyệt toàn bộ**.

In [7]:
class MyHashMapList:
    def __init__(self):
        self.pairs = []

    def put(self, key, value):
        for pair in self.pairs:
            if pair[0] == key:
                pair[1] = value          # cập nhật
                return
        self.pairs.append([key, value])  # thêm mới

    def get(self, key):
        for k, v in self.pairs:
            if k == key:
                return v
        return -1

    def remove(self, key):
        for i, (k, _) in enumerate(self.pairs):
            if k == key:
                self.pairs.pop(i)
                return

print(run_ops(MyHashMapList, OPS, ARGS))
print(run_ops(MyHashMapList, OPS, ARGS) == EXP)

[None, None, None, 1, -1, None, 1, None, -1]
True


**Nút thắt:** mỗi thao tác `O(n)` (`n` = số key). Với `10^4` lời gọi, xấu nhất cỡ `5·10^7` phép so sánh. Ta cần "nhảy thẳng" tới chỗ có thể chứa key.

## 4. Cách 2 — Mảng trực tiếp (direct addressing)

Vì `0 <= key <= 10^6`: list `vals` dài `10^6 + 1`, `vals[key]` là value, `-1` nghĩa là "không có" (an toàn vì value `>= 0`).

In [8]:
class MyHashMapDirect:
    def __init__(self):
        self.vals = [-1] * (10**6 + 1)
    def put(self, key, value):
        self.vals[key] = value
    def get(self, key):
        return self.vals[key]
    def remove(self, key):
        self.vals[key] = -1

print(run_ops(MyHashMapDirect, OPS, ARGS) == EXP)

True


- Ưu: `O(1)` thật sự, cực ngắn.
- Nhược: luôn tốn `10^6 + 1` ô (~8 MB với list Python) dù chỉ có vài key; không dùng được khi key là chuỗi hay số rất lớn.

Có thể tiết kiệm bằng `array('i', ...)` (mỗi ô 4 byte) từ module chuẩn `array`.

In [9]:
import sys
from array import array
a_list = [-1] * (10**6 + 1)
a_arr = array('i', [-1]) * (10**6 + 1)
print("list :", sys.getsizeof(a_list) // 1024, "KB")
print("array:", sys.getsizeof(a_arr) // 1024, "KB")

list : 7812 KB
array: 3906 KB


## 5. Insight — băm + chaining các cặp

Ghép ưu điểm của hai cách:
1. Băm `key % M` (với `M` nguyên tố, ví dụ `1009` hoặc `2069`) để **chọn bucket** trong `O(1)`.
2. Mỗi bucket là list nhỏ các cặp `[key, value]` — chỉ duyệt **trong bucket**.

Nếu key phân bố đều, mỗi bucket chứa khoảng `α = n / M` cặp (load factor) → thao tác `O(1 + α)` trung bình.

Ba phương thức cùng một "khuôn": **tìm cặp có key trong bucket** rồi:
- `put`: có → sửa value; không → append.
- `get`: có → trả value; không → `-1`.
- `remove`: có → xóa cặp; không → không làm gì.

In [10]:
M = 1009
for key in [1, 1010, 2019, 5]:
    print(f"key {key:>4} -> bucket {key % M}")
print("=> 1, 1010, 2019 va chạm ở bucket 1, phải phân biệt bằng cách so key trong bucket")

key    1 -> bucket 1
key 1010 -> bucket 1
key 2019 -> bucket 1
key    5 -> bucket 5
=> 1, 1010, 2019 va chạm ở bucket 1, phải phân biệt bằng cách so key trong bucket


## 6. Cách 3 — Băm + chaining bằng list (lời giải chính)

Viết một hàm phụ `_find(bucket, key)` trả về **chỉ số** của cặp trong bucket (hoặc `-1`), để 3 phương thức dùng chung và không lặp code.

In [11]:
class MyHashMapChain:
    def __init__(self):
        self.M = 1009
        self.buckets = [[] for _ in range(self.M)]

    def _find(self, bucket, key):
        for i, (k, _) in enumerate(bucket):
            if k == key:
                return i
        return -1

    def put(self, key, value):
        bucket = self.buckets[key % self.M]
        i = self._find(bucket, key)
        if i >= 0:
            bucket[i][1] = value         # cập nhật
        else:
            bucket.append([key, value])  # thêm mới

    def get(self, key):
        bucket = self.buckets[key % self.M]
        i = self._find(bucket, key)
        return bucket[i][1] if i >= 0 else -1

    def remove(self, key):
        bucket = self.buckets[key % self.M]
        i = self._find(bucket, key)
        if i >= 0:
            bucket.pop(i)

print(run_ops(MyHashMapChain, OPS, ARGS))
print(run_ops(MyHashMapChain, OPS, ARGS) == EXP)

[None, None, None, 1, -1, None, 1, None, -1]
True


`bucket[i][1] = value`: `bucket[i]` là cặp `[key, value]`, `[1]` là value. Vì `bucket` là tham chiếu tới list trong `self.buckets`, sửa ở đây là sửa thẳng vào bảng.

`bucket.pop(i)` tốn `O(len(bucket))` vì dịch phần tử sau `i`. Bucket ngắn nên không đáng kể. (Mẹo khác: đổi chỗ phần tử `i` với phần tử cuối rồi `pop()` — `O(1)`, vì thứ tự trong bucket không quan trọng.)

## 7. TRACE — nhìn bucket thay đổi

Dùng `M = 5` để thấy va chạm: `1`, `6`, `11` đều vào bucket 1.

| Lệnh | bucket | Tìm thấy? | Hành động | Bucket 1 sau lệnh |
|---|:---:|:---:|---|---|
| put(1, 10) | 1 | không | append | [[1,10]] |
| put(6, 60) | 1 | không | append (va chạm) | [[1,10],[6,60]] |
| put(1, 15) | 1 | có (i=0) | cập nhật | [[1,15],[6,60]] |
| get(11) | 1 | không | trả -1 | không đổi |
| get(6) | 1 | có (i=1) | trả 60 | không đổi |
| remove(1) | 1 | có (i=0) | pop(0) | [[6,60]] |
| get(1) | 1 | không | trả -1 | [[6,60]] |

In [12]:
class MyHashMapTrace(MyHashMapChain):
    def __init__(self, M=5):
        self.M = M
        self.buckets = [[] for _ in range(M)]
    def state(self):
        return {i: bk for i, bk in enumerate(self.buckets) if bk}
    def put(self, key, value):
        bk = self.buckets[key % self.M]
        i = self._find(bk, key)
        act = f"có (i={i}) -> cập nhật" if i >= 0 else "không -> append"
        super().put(key, value)
        print(f"put({key},{value}):".ljust(14), f"bucket {key % self.M}, {act:22} | {self.state()}")
    def get(self, key):
        res = super().get(key)
        print(f"get({key}):".ljust(14), f"bucket {key % self.M}, duyệt {self.buckets[key % self.M]} -> {res}")
        return res
    def remove(self, key):
        bk = self.buckets[key % self.M]
        i = self._find(bk, key)
        act = f"có (i={i}) -> pop" if i >= 0 else "không -> bỏ qua"
        super().remove(key)
        print(f"remove({key}):".ljust(14), f"bucket {key % self.M}, {act:22} | {self.state()}")

m = MyHashMapTrace(5)
m.put(1, 10); m.put(6, 60); m.put(1, 15)
m.get(11); m.get(6)
m.remove(1); m.get(1)

put(1,10):     bucket 1, không -> append        | {1: [[1, 10]]}
put(6,60):     bucket 1, không -> append        | {1: [[1, 10], [6, 60]]}
put(1,15):     bucket 1, có (i=0) -> cập nhật   | {1: [[1, 15], [6, 60]]}
get(11):       bucket 1, duyệt [[1, 15], [6, 60]] -> -1
get(6):        bucket 1, duyệt [[1, 15], [6, 60]] -> 60
remove(1):     bucket 1, có (i=0) -> pop        | {1: [[6, 60]]}
get(1):        bucket 1, duyệt [[6, 60]] -> -1


-1

`super().put(key, value)` gọi phương thức `put` của **lớp cha** (`MyHashMapChain`) — lớp trace chỉ thêm phần `print`, không viết lại logic.

Trace kịch bản mẫu của đề:

In [13]:
_ = run_ops(lambda: MyHashMapTrace(5), OPS, ARGS)

put(1,1):      bucket 1, không -> append        | {1: [[1, 1]]}
put(2,2):      bucket 2, không -> append        | {1: [[1, 1]], 2: [[2, 2]]}
get(1):        bucket 1, duyệt [[1, 1]] -> 1
get(3):        bucket 3, duyệt [] -> -1
put(2,1):      bucket 2, có (i=0) -> cập nhật   | {1: [[1, 1]], 2: [[2, 1]]}
get(2):        bucket 2, duyệt [[2, 1]] -> 1
remove(2):     bucket 2, có (i=0) -> pop        | {1: [[1, 1]]}
get(2):        bucket 2, duyệt [] -> -1


## 8. Code nộp LeetCode

In [14]:
class MyHashMap:

    def __init__(self):
        self.size = 2069
        self.buckets = [[] for _ in range(self.size)]

    def _locate(self, key: int):
        bucket = self.buckets[key % self.size]
        for i, pair in enumerate(bucket):
            if pair[0] == key:
                return bucket, i
        return bucket, -1

    def put(self, key: int, value: int) -> None:
        bucket, i = self._locate(key)
        if i >= 0:
            bucket[i][1] = value
        else:
            bucket.append([key, value])

    def get(self, key: int) -> int:
        bucket, i = self._locate(key)
        return bucket[i][1] if i >= 0 else -1

    def remove(self, key: int) -> None:
        bucket, i = self._locate(key)
        if i >= 0:
            bucket[i] = bucket[-1]      # đổi với phần tử cuối
            bucket.pop()                # rồi pop O(1)

print(run_ops(MyHashMap, OPS, ARGS) == EXP)

True


`_locate` trả về **hai giá trị** (tuple `(bucket, i)`), người gọi unpack `bucket, i = ...`. Như vậy mỗi phương thức chỉ băm và tìm **một lần**.

Bản mảng trực tiếp (ngắn, `O(1)` chắc chắn):

```python
class MyHashMap:
    def __init__(self):
        self.vals = [-1] * (10**6 + 1)
    def put(self, key: int, value: int) -> None:
        self.vals[key] = value
    def get(self, key: int) -> int:
        return self.vals[key]
    def remove(self, key: int) -> None:
        self.vals[key] = -1
```

## 9. Cách 4 — Băm + linked list với dummy node

Cách kinh điển trong giáo trình: mỗi bucket là danh sách liên kết; node lưu `key`, `val`, `next`. Dummy node đầu bucket giúp xóa đồng nhất.

In [15]:
class ListNode:
    def __init__(self, key=-1, val=-1, nxt=None):
        self.key, self.val, self.next = key, val, nxt

class MyHashMapLinked:
    def __init__(self, size=1009):
        self.size = size
        self.heads = [ListNode() for _ in range(size)]

    def _prev(self, key):
        p = self.heads[key % self.size]
        while p.next and p.next.key != key:
            p = p.next
        return p

    def put(self, key, value):
        p = self._prev(key)
        if p.next:
            p.next.val = value
        else:
            p.next = ListNode(key, value)

    def get(self, key):
        p = self._prev(key)
        return p.next.val if p.next else -1

    def remove(self, key):
        p = self._prev(key)
        if p.next:
            p.next = p.next.next

print(run_ops(MyHashMapLinked, OPS, ARGS) == EXP)

True


`self.key, self.val, self.next = key, val, nxt`: gán nhiều thuộc tính một dòng (tuple unpacking).

## 10. Vì sao đúng?

**Invariant:** *Map logic chứa cặp `(k, v)` ⇔ bucket `k % M` chứa **đúng một** cặp `[k, v]`; không bucket nào chứa cặp có key `k` khác.*

- Khởi tạo: mọi bucket rỗng, map rỗng ✔.
- `put(k, v)`: chỉ chạm bucket `k % M`. Nếu đã có cặp key `k` → sửa value của chính cặp đó (vẫn một cặp) ✔; nếu chưa → thêm một cặp ✔.
- `remove(k)`: nếu có (đúng một cặp) → xóa nó → không còn cặp key `k` ✔. Phép "đổi với cuối rồi pop" chỉ thay đổi thứ tự trong bucket, không làm mất/nhân bản cặp khác ✔.
- `get(k)`: theo invariant chỉ cần tìm trong bucket `k % M`; tìm thấy thì value đó là value mới nhất ✔.

Nếu `put` luôn `append` (không kiểm tra key), bucket có thể chứa hai cặp cùng key → `get` trả value **cũ** → sai (xem lỗi 1).

## 11. Độ phức tạp và so sánh

`n` = số key đang lưu, `M` = số bucket, `U = 10^6 + 1`.

| Cách | put / get / remove | Bộ nhớ | Nhận xét |
|---|---|---|---|
| 1. List các cặp | `O(n)` | `O(n)` | Baseline |
| 2. Mảng trực tiếp | `O(1)` | `O(U)` | Miền key nhỏ mới dùng được |
| 3. Băm + chaining (list cặp) | `O(1 + n/M)` TB, `O(n)` xấu nhất | `O(M + n)` | **Nên nộp** |
| 4. Băm + linked list | như 3 | `O(M + n)` | Kinh điển, dài hơn |

Xấu nhất xảy ra khi mọi key cùng bucket (ví dụ toàn bội số của `M`).

In [16]:
import random, time
rng = random.Random(706)
ops = []
for _ in range(10**4):
    op = rng.choice(["put", "get", "remove"])
    k = rng.randint(0, 10**6)
    ops.append((op, (k, rng.randint(0, 10**6)) if op == "put" else (k,)))
for cls in [MyHashMapList, MyHashMapDirect, MyHashMap, MyHashMapLinked]:
    obj = cls()
    t0 = time.perf_counter()
    for op, a in ops:
        getattr(obj, op)(*a)
    print(f"{cls.__name__:16} 10^4 thao tác: {time.perf_counter() - t0:.4f}s")

MyHashMapList    10^4 thao tác: 0.4764s
MyHashMapDirect  10^4 thao tác: 0.0030s
MyHashMap        10^4 thao tác: 0.0056s
MyHashMapLinked  10^4 thao tác: 0.0056s


In [17]:
bad = MyHashMap()
t0 = time.perf_counter()
for i in range(3000):
    bad.put(i * 2069, i)                 # cố ý: mọi key cùng bucket 0
for i in range(3000):
    bad.get(i * 2069)
print("bucket 0 dài:", len(bad.buckets[0]), "| thời gian xấu nhất:", round(time.perf_counter() - t0, 3), "s")

bucket 0 dài: 3000 | thời gian xấu nhất: 0.313 s


## 12. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:34} PASS")

check("mẫu đề", run_ops(MyHashMap, OPS, ARGS), EXP)
m = MyHashMap()
m.put(0, 0)
check("key = 0, value = 0 (không nhầm với -1)", m.get(0), 0)
m.put(10**6, 10**6)
check("key = 10^6", m.get(10**6), 10**6)
m.put(5, 1); m.put(5, 2); m.put(5, 3)
check("cập nhật nhiều lần", m.get(5), 3)
m.remove(5)
check("remove sau nhiều lần put", m.get(5), -1)
m.remove(12345)
check("remove key không có", m.get(0), 0)
m.put(7, 70); m.put(7 + 2069, 71); m.put(7 + 2 * 2069, 72)
m.remove(7 + 2069)
check("va chạm: xóa cặp giữa", (m.get(7), m.get(7 + 2069), m.get(7 + 2 * 2069)), (70, -1, 72))

mẫu đề                             PASS
key = 0, value = 0 (không nhầm với -1) PASS
key = 10^6                         PASS
cập nhật nhiều lần                 PASS
remove sau nhiều lần put           PASS
remove key không có                PASS
va chạm: xóa cặp giữa              PASS


In [19]:
impls = [MyHashMapList, MyHashMapDirect, MyHashMapChain, MyHashMap, MyHashMapLinked]
for trial in range(200):
    objs = [c() for c in impls]
    oracle = {}            # dict có sẵn CHỈ dùng làm đáp án chuẩn để kiểm thử
    pool = [rng.randint(0, 10**6) for _ in range(5)] + [1009 * i + 3 for i in range(3)] + [2069 * i + 3 for i in range(3)]
    for _ in range(120):
        op, k = rng.choice(["put", "get", "remove"]), rng.choice(pool)
        if op == "put":
            v = rng.randint(0, 3)
            oracle[k] = v
            args, exp = (k, v), None
        elif op == "get":
            args, exp = (k,), oracle.get(k, -1)
        else:
            oracle.pop(k, None)
            args, exp = (k,), None
        for o in objs:
            assert getattr(o, op)(*args) == exp, (type(o).__name__, op, args)
print("200 kịch bản ngẫu nhiên (có va chạm, value 0): 5 cài đặt khớp dict chuẩn")

200 kịch bản ngẫu nhiên (có va chạm, value 0): 5 cài đặt khớp dict chuẩn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — `put` luôn append, không cập nhật

In [20]:
class WrongAppend(MyHashMapChain):
    def put(self, key, value):
        self.buckets[key % self.M].append([key, value])   # SAI

print("sai :", run_ops(WrongAppend, OPS, ARGS))
print("đúng:", EXP)

sai : [None, None, None, 1, -1, None, 2, None, 1]
đúng: [None, None, None, 1, -1, None, 1, None, -1]


Vì sao sai: sau `put(2, 1)` bucket có `[[2,2],[2,1]]`; `get(2)` tìm thấy cặp **đầu** → trả `2` (value cũ). Và `remove(2)` chỉ xóa một cặp → `get(2)` vẫn ra giá trị. Sửa: tìm key trước, có thì cập nhật.

### Lỗi 2 — Dùng `0` làm "không có" trong mảng trực tiếp

In [21]:
class WrongZero(MyHashMapDirect):
    def __init__(self):
        self.vals = [0] * (10**6 + 1)
    def get(self, key):
        return self.vals[key] if self.vals[key] != 0 else -1

w = WrongZero()
w.put(3, 0)
print("sai: get(3) sau put(3, 0) =", w.get(3), "| đúng: 0")

sai: get(3) sau put(3, 0) = -1 | đúng: 0


Vì sao sai: value `0` hợp lệ nhưng bị hiểu là "không có". Sửa: khởi tạo `-1` (giá trị nằm **ngoài** miền value).

### Lỗi 3 — Xóa phần tử trong lúc đang duyệt mà không `break`

In [22]:
bucket = [[1, 10], [1, 11], [6, 60]]   # giả sử (do lỗi khác) có 2 cặp key 1
for i, (k, v) in enumerate(bucket):
    if k == 1:
        bucket.pop(i)                   # không break -> bỏ sót phần tử kế tiếp
print(bucket)

[[1, 11], [6, 60]]


Vì sao sai: sau `pop(0)`, phần tử `[1, 11]` dịch lên vị trí 0 nhưng vòng lặp đã sang `i = 1` → bỏ sót. Với invariant "mỗi key một cặp", chỉ cần xóa một lần rồi `break`/`return`.

### Lỗi 4 — `[[]] * M`
Tất cả bucket là một list chung → `put` vào key này có thể "thấy" ở mọi bucket; mất ý nghĩa băm và thành `O(n)`.

### Lỗi 5 — Quên trả `-1` khi không tìm thấy
Hàm Python không có `return` sẽ trả `None`, LeetCode báo sai (`null` thay vì `-1`).

In [23]:
def wrong_get(bucket, key):
    for k, v in bucket:
        if k == key:
            return v
    # quên return -1

print(wrong_get([[1, 10]], 5))

None


## 14. Mẹo nhận dạng pattern và biến thể

- Khi key là **số nguyên trong miền nhỏ** (ký tự, điểm số, `0..10^6`): mảng đếm/mảng trực tiếp thay cho dict, nhanh và đơn giản (ví dụ 242 Valid Anagram dùng list 26 ô).
- Khi cần map tổng quát: băm + xử lý va chạm (chaining hoặc open addressing).
- Mọi thao tác HashMap đều là "**tìm cặp theo key trong bucket**" + một hành động → viết một hàm `_locate` dùng chung.

Bài liên quan: 705 Design HashSet, 146 LRU Cache (HashMap + doubly linked list), 380 Insert Delete GetRandom O(1), 1 Two Sum (dùng HashMap).

### 14.1. Biến thể A — key là chuỗi: tự viết hàm băm đa thức

In [24]:
def poly_hash(s: str, M: int = 1009, base: int = 31) -> int:
    h = 0
    for ch in s:
        h = (h * base + ord(ch)) % M
    return h

for w in ["apple", "banana", "elppa", "a", "b"]:
    print(f"{w:7} -> bucket {poly_hash(w)}")

apple   -> bucket 419
banana  -> bucket 256
elppa   -> bucket 572
a       -> bucket 97
b       -> bucket 98


`ord(ch)` trả mã số của ký tự (`ord('a') = 97`). Nhân với `base` theo vị trí giúp `"apple"` và `"elppa"` (cùng ký tự, khác thứ tự) thường cho bucket khác nhau.

### 14.2. Biến thể B — `get` với giá trị mặc định và `put_if_absent`

In [25]:
class MyHashMapPlus(MyHashMap):
    def get_default(self, key, default):
        bucket, i = self._locate(key)
        return bucket[i][1] if i >= 0 else default
    def put_if_absent(self, key, value):
        bucket, i = self._locate(key)
        if i < 0:
            bucket.append([key, value])
            return value
        return bucket[i][1]

p = MyHashMapPlus()
print(p.put_if_absent(4, 40), p.put_if_absent(4, 99), p.get_default(5, "không có"))

40 40 không có


## 15. Bài tập tự luyện

**Bài 1 (tính tay).** `M = 7`. Sau `put(3,30), put(10,100), put(3,31), put(17,170), remove(10)`, bucket 3 chứa gì? Gán vào `ex1_bucket3`.

**Bài 2 (đoán output).** Với `MyHashMapDirect`: `put(9, 0); get(9); remove(9); get(9)` → hai giá trị `get` là gì? Gán tuple vào `ex2_gets`.

**Bài 3 (TODO).** Tự viết `Ex3HashMap` theo cách chaining (không xem code trên).

**Bài 4 (biến thể).** `Ex4Counter`: `inc(key)`, `dec(key)` (không xuống dưới 0, về 0 thì xóa), `count(key)` — không dùng `dict`.

**Bài 5 (biến thể).** `ex5_two_sum(nums, target)` (LeetCode 1) dùng **chính `MyHashMap` của bạn** thay cho `dict` (giả sử `0 <= nums[i] <= 10^6`).

**Bài 6 (mini challenge).** `Ex6StrMap`: HashMap với key là **chuỗi**, value số nguyên, có `put/get/remove` và tự **resize** gấp đôi khi `n / M > 1`.

In [26]:
ex1_bucket3 = None   # TODO
ex2_gets = None      # TODO

class Ex3HashMap:
    def __init__(self): pass                # TODO
    def put(self, key, value): pass         # TODO
    def get(self, key): pass                # TODO
    def remove(self, key): pass             # TODO

# assert run_ops(Ex3HashMap, OPS, ARGS) == EXP

In [27]:
class Ex4Counter:
    def __init__(self): pass        # TODO
    def inc(self, key): pass        # TODO
    def dec(self, key): pass        # TODO
    def count(self, key): pass      # TODO

def ex5_two_sum(nums, target):
    # TODO: dùng MyHashMap: value -> index
    pass

class Ex6StrMap:
    def __init__(self): pass                # TODO
    def put(self, key, value): pass         # TODO
    def get(self, key): pass                # TODO
    def remove(self, key): pass             # TODO

# c = Ex4Counter(); c.inc(3); c.inc(3); c.dec(3); assert c.count(3) == 1
# assert ex5_two_sum([2, 7, 11, 15], 9) == [0, 1]
# s = Ex6StrMap(); s.put("a", 1); s.put("a", 2); assert s.get("a") == 2

## 16. Gợi ý 3 mức

- **Mức 1:** Mọi thao tác = băm chọn bucket → tìm cặp theo key → hành động.
- **Mức 2:** Bài 4 = HashMap với value là số đếm. Bài 5: duyệt `i, x`; `need = target - x`; nếu `need >= 0` và `get(need) != -1` thì trả `[get(need), i]`; sau đó `put(x, i)`. Bài 6: băm chuỗi bằng `poly_hash`; khi resize, tạo mảng bucket mới gấp đôi và **chèn lại mọi cặp** (vị trí bucket thay đổi vì `M` đổi).
- **Mức 3:** Bài 6 resize:
  ```text
  old = self.buckets; self.M *= 2
  self.buckets = [[] for _ in range(self.M)]
  for bk in old: for k, v in bk: self.buckets[h(k)].append([k, v])
  ```

## 17. Đáp án tham khảo

Bài 1: `3, 10, 17` đều `% 7 = 3`. Sau các lệnh: `[[3,30]]` → `[[3,30],[10,100]]` → `[[3,31],[10,100]]` → thêm `[17,170]` → xóa 10. Với cách `pop(i)` giữ thứ tự: `[[3, 31], [17, 170]]`.
Bài 2: `(0, -1)`.

In [28]:
ans_ex1_bucket3 = [[3, 31], [17, 170]]
ans_ex2_gets = (0, -1)

class AnsEx3HashMap:
    def __init__(self):
        self.M = 1009
        self.b = [[] for _ in range(self.M)]
    def put(self, key, value):
        bk = self.b[key % self.M]
        for pair in bk:
            if pair[0] == key:
                pair[1] = value
                return
        bk.append([key, value])
    def get(self, key):
        for k, v in self.b[key % self.M]:
            if k == key:
                return v
        return -1
    def remove(self, key):
        bk = self.b[key % self.M]
        for i, (k, _) in enumerate(bk):
            if k == key:
                bk.pop(i)
                return

class AnsEx4Counter:
    def __init__(self):
        self.m = AnsEx3HashMap()
    def inc(self, key):
        c = self.m.get(key)
        self.m.put(key, 1 if c == -1 else c + 1)
    def dec(self, key):
        c = self.m.get(key)
        if c == 1:
            self.m.remove(key)
        elif c > 1:
            self.m.put(key, c - 1)
    def count(self, key):
        c = self.m.get(key)
        return 0 if c == -1 else c

def ans_ex5_two_sum(nums, target):
    seen = MyHashMap()
    for i, x in enumerate(nums):
        need = target - x
        if 0 <= need <= 10**6:
            j = seen.get(need)
            if j != -1:
                return [j, i]
        seen.put(x, i)
    return []

class AnsEx6StrMap:
    def __init__(self):
        self.M, self.n = 4, 0
        self.b = [[] for _ in range(self.M)]
    def _h(self, key):
        return poly_hash(key, self.M)
    def put(self, key, value):
        bk = self.b[self._h(key)]
        for pair in bk:
            if pair[0] == key:
                pair[1] = value
                return
        bk.append([key, value]); self.n += 1
        if self.n / self.M > 1:
            old = self.b
            self.M *= 2
            self.b = [[] for _ in range(self.M)]
            for obk in old:
                for k, v in obk:
                    self.b[self._h(k)].append([k, v])
    def get(self, key):
        for k, v in self.b[self._h(key)]:
            if k == key:
                return v
        return -1
    def remove(self, key):
        bk = self.b[self._h(key)]
        for i, (k, _) in enumerate(bk):
            if k == key:
                bk.pop(i); self.n -= 1
                return

h7 = AnsEx3HashMap(); h7.M = 7; h7.b = [[] for _ in range(7)]
h7.put(3, 30); h7.put(10, 100); h7.put(3, 31); h7.put(17, 170); h7.remove(10)
assert h7.b[3] == ans_ex1_bucket3
dm = MyHashMapDirect(); dm.put(9, 0); g1 = dm.get(9); dm.remove(9)
assert (g1, dm.get(9)) == ans_ex2_gets
assert run_ops(AnsEx3HashMap, OPS, ARGS) == EXP
c = AnsEx4Counter(); c.inc(3); c.inc(3); c.dec(3)
assert c.count(3) == 1
c.dec(3); c.dec(3)
assert c.count(3) == 0
assert ans_ex5_two_sum([2, 7, 11, 15], 9) == [0, 1]
assert ans_ex5_two_sum([3, 2, 4], 6) == [1, 2]
s = AnsEx6StrMap(); s.put("a", 1); s.put("a", 2)
assert s.get("a") == 2
words = [f"w{i}" for i in range(500)]
for i, w in enumerate(words):
    s.put(w, i)
assert all(s.get(w) == i for i, w in enumerate(words)) and s.M >= 256
for w in words[::2]:
    s.remove(w)
assert s.get("w0") == -1 and s.get("w1") == 1
print("Đáp án tham khảo: PASS toàn bộ | Ex6 đã resize tới M =", s.M)

Đáp án tham khảo: PASS toàn bộ | Ex6 đã resize tới M = 512


## 18. Template nhớ nhanh và checklist

```python
class MyHashMap:
    def __init__(self):
        self.size = 2069
        self.buckets = [[] for _ in range(self.size)]
    def _locate(self, key):
        bucket = self.buckets[key % self.size]
        for i, pair in enumerate(bucket):
            if pair[0] == key:
                return bucket, i
        return bucket, -1
    def put(self, key, value):
        bucket, i = self._locate(key)
        if i >= 0: bucket[i][1] = value
        else: bucket.append([key, value])
    def get(self, key):
        bucket, i = self._locate(key)
        return bucket[i][1] if i >= 0 else -1
    def remove(self, key):
        bucket, i = self._locate(key)
        if i >= 0:
            bucket[i] = bucket[-1]; bucket.pop()
```

Câu thần chú: **"Băm → tìm cặp → (put: sửa hoặc thêm | get: trả hoặc -1 | remove: xóa nếu có)."**

### Checklist trước khi submit
- [ ] Không dùng `dict`/`set` trong lời giải.
- [ ] `put` cập nhật khi key đã có (không thêm cặp trùng key).
- [ ] `get` trả `-1` khi không có; value `0` vẫn trả `0`.
- [ ] Bucket tạo bằng list comprehension.
- [ ] Test key/value `0`, key `10^6`, các key va chạm, xóa key không tồn tại.
- [ ] Nói được: TB `O(1 + n/M)`, xấu nhất `O(n)`, bộ nhớ `O(M + n)`.